# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
Carlos Muñiz Lara

# Create SparkSession

In [1]:
from spark_utils import SparkUtils

su = SparkUtils()
su.spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 01:00:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Songs recommednation

In [2]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [3]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5


Number of users (m):3


In [4]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [5]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [6]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

+-------+------------------------------------------------+
|user_id|recommendations                                 |
+-------+------------------------------------------------+
|1      |[{2, 4.9683604}, {5, 4.8511505}, {1, 3.9393601}]|
|2      |[{3, 3.9399467}, {2, 2.9783492}, {4, 2.9044724}]|
|3      |[{3, 4.831993}, {4, 3.4459817}, {2, 2.875986}]  |
+-------+------------------------------------------------+



In [7]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [8]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.9683604|
|1      |song e|4.8511505|
|1      |song a|3.9393601|
|2      |song c|3.9399467|
|2      |song b|2.9783492|
|2      |song d|2.9044724|
|3      |song c|4.831993 |
|3      |song d|3.4459817|
|3      |song b|2.875986 |
+-------+------+---------+



In [9]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9393601 |
|1      |2      |5     |4.9683604 |
|1      |5      |5     |4.8511505 |
|2      |2      |3     |2.9783492 |
|3      |1      |2     |1.9603102 |
|3      |3      |5     |4.831993  |
|3      |5      |1     |1.0399308 |
|2      |3      |4     |3.9399467 |
|2      |4      |3     |2.9044724 |
+-------+-------+------+----------+



In [10]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.0890887102319288


# Lab 12: Building a Recommendation System with ALS 

In [13]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

In [20]:

als = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

model = als.fit(movies_ratings_df)


user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)
movies = [
    (1, "9 Queens"),
    (2, "Batman Begins"),
    (3, "Star Wars: Episode IV - A New Hope"),
    (4, "The Lord of the Rings: The Fellowship of the Ring"),
    (5, "Project Hail Mary")]

movies_schema_df = SparkUtils.generate_schema([("movie_id", "int"), ("title", "string")])
movies_df = su.spark.createDataFrame(movies, movies_schema_df)




+-------+---------------------------------------------------+
|user_id|recommendations                                    |
+-------+---------------------------------------------------+
|0      |[{92, 2.6372044}, {2, 2.372304}, {62, 2.2646565}]  |
|10     |[{92, 2.8236146}, {2, 2.7020485}, {93, 2.6298938}] |
|20     |[{22, 3.538786}, {68, 3.1066718}, {94, 3.094256}]  |
|1      |[{22, 2.8790956}, {68, 2.598576}, {77, 2.5562754}] |
|11     |[{32, 5.0153604}, {30, 4.7513}, {18, 4.6264906}]   |
|21     |[{29, 4.3093877}, {52, 4.2315116}, {76, 3.702759}] |
|22     |[{51, 4.4479766}, {75, 4.423848}, {74, 4.092779}]  |
|2      |[{93, 4.2578244}, {83, 4.1887317}, {8, 4.0659637}] |
|12     |[{46, 5.777978}, {55, 4.790728}, {49, 4.5164623}]  |
|23     |[{46, 5.56972}, {55, 4.716812}, {32, 4.670099}]    |
|3      |[{30, 4.12416}, {69, 3.8931205}, {51, 3.8512409}]  |
|13     |[{74, 2.6669812}, {93, 2.632752}, {29, 2.5154982}] |
|24     |[{29, 4.430447}, {52, 4.425605}, {30, 3.9473932}]  |
|4      

In [22]:
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(movies_df, recommendations.rec.movie_id == movies_df.movie_id).select("user_id", "title", "rec.rating")

# Show user-movie recommendations with titles
recommendations.show(truncate=False)

+-------+-------------+---------+
|user_id|title        |rating   |
+-------+-------------+---------+
|0      |Batman Begins|2.372304 |
|10     |Batman Begins|2.7020485|
+-------+-------------+---------+



## Predictions

In [23]:
predictions = model.transform(movies_ratings_df)
predictions.show(truncate=False)

+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|0.9738388 |
|22     |3       |2     |1424380312|1.6255691 |
|22     |5       |2     |1424380312|2.085125  |
|22     |6       |2     |1424380312|2.296648  |
|22     |9       |1     |1424380312|1.587081  |
|22     |10      |1     |1424380312|1.4329246 |
|22     |11      |1     |1424380312|1.27851   |
|22     |13      |1     |1424380312|1.6188018 |
|22     |14      |1     |1424380312|1.3702021 |
|22     |16      |1     |1424380312|0.69885296|
|22     |18      |3     |1424380312|3.0429049 |
|22     |19      |1     |1424380312|1.4495806 |
|22     |22      |5     |1424380312|4.0900083 |
|22     |25      |1     |1424380312|0.99060917|
|22     |26      |1     |1424380312|1.142211  |
|22     |29      |3     |1424380312|3.2190433 |
|22     |30      |5     |1424380312|3.9957788 |
|22     |32      |4     |1424380312|3.14

## Test ML Model

In [24]:
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.5696929650471643


## Persist the model

In [27]:
model_path = "/opt/spark/work-dir/models/als_movies"

model.write().overwrite().save(model_path)
print(f"Model saved at: {model_path}")

Model saved at: /opt/spark/work-dir/models/als_movies


In [28]:
su.spark.stop()